# Pertemuan 4 — Pencarian Tanpa Informasi (Uninformed Search)
**PIF403 Pengantar Kecerdasan Buatan · Prodi Informatika Universitas Adzkia**

Minggu lalu kita memformulasikan masalah menjadi ruang keadaan. Sekarang kita menelusuri ruang itu secara sistematis **tanpa petunjuk apa pun** tentang letak tujuan. Lima strategi dibahas: BFS, DFS, depth-limited, iterative deepening, dan uniform-cost.

**Setelah menyelesaikan notebook ini Anda mampu:**
1. mengimplementasikan dan menelusuri BFS, DFS, DLS, IDS, dan UCS;
2. mengukur jumlah node yang diekspansi dan ukuran frontier;
3. membandingkan kelima strategi dari sisi *completeness*, *optimality*, waktu, dan memori.

**Cara pakai:** simpan salinan ke Drive, isi identitas, lalu jalankan dari atas. Bagian **Latihan** adalah tahap pertama asesmen A2 (dikumpulkan bersama notebook pertemuan 5).

In [ ]:
NAMA = "tulis nama Anda"
NIM = "tulis NIM Anda"
print(f"Notebook P4 dikerjakan oleh: {NAMA} ({NIM})")

## 1. Menyiapkan data dan formulasi masalah
Unggah `peta_sumbar.csv` dan `kota_sumbar.csv` (berkas yang sama dengan pertemuan 3).

In [ ]:
import os

BERKAS = ["peta_sumbar.csv", "kota_sumbar.csv"]
if not all(os.path.exists(b) for b in BERKAS):
    from google.colab import files
    print("Unggah berkas:", ", ".join(BERKAS))
    files.upload()
for b in BERKAS:
    assert os.path.exists(b), f"Berkas {b} belum diunggah. Jalankan ulang sel ini."

import pandas as pd, networkx as nx, matplotlib.pyplot as plt
import heapq, random, time
from collections import deque

jalan = pd.read_csv("peta_sumbar.csv"); kota = pd.read_csv("kota_sumbar.csv")
G = nx.Graph()
for _, r in jalan.iterrows():
    G.add_edge(r.kota_asal, r.kota_tujuan, jarak=int(r.jarak_km))
POSISI = {r.kota: (r.x_gambar, r.y_gambar) for _, r in kota.iterrows()}
print(G.number_of_nodes(), "kota,", G.number_of_edges(), "ruas jalan")

In [ ]:
# Kerangka masalah dari pertemuan 3
class Masalah:
    def __init__(self, awal): self.awal = awal
    def aksi(self, s): raise NotImplementedError
    def hasil(self, s, a): raise NotImplementedError
    def tujuan(self, s): raise NotImplementedError
    def biaya(self, s, a, s2): return 1

class MasalahRute(Masalah):
    def __init__(self, graf, awal, kota_tujuan):
        super().__init__(awal); self.graf, self.kota_tujuan = graf, kota_tujuan
    def aksi(self, s): return sorted(self.graf.neighbors(s))      # urutan alfabet
    def hasil(self, s, a): return a
    def tujuan(self, s): return s == self.kota_tujuan
    def biaya(self, s, a, s2): return self.graf[s][s2]["jarak"]

rute = MasalahRute(G, "Padang", "Payakumbuh")

## 2. Node, frontier, dan himpunan *reached*

Algoritma pencarian membangun **pohon pencarian** di atas graf ruang keadaan. Tiga istilah kunci:

- **Node**: catatan pembukuan pencarian. Isinya status, node induk, aksi yang menghasilkannya, dan biaya jalur sejauh ini `g`. Satu status dapat muncul pada beberapa node.
- **Frontier**: node yang sudah dibangkitkan tetapi belum diekspansi. *Struktur data frontier inilah yang membedakan satu algoritma dari yang lain.*
- **Reached**: status yang sudah pernah dicapai, supaya pencarian tidak berputar-putar.

| Algoritma | Frontier | Node yang diambil lebih dulu |
|---|---|---|
| BFS | antrean FIFO | yang paling lama menunggu (paling dangkal) |
| DFS | tumpukan LIFO | yang paling baru masuk (paling dalam) |
| UCS | antrean prioritas | yang `g`-nya paling kecil |

In [ ]:
class Node:
    def __init__(self, status, induk=None, aksi=None, g=0):
        self.status, self.induk, self.aksi, self.g = status, induk, aksi, g
        self.kedalaman = 0 if induk is None else induk.kedalaman + 1
    def jalur(self):
        n, hasil = self, []
        while n: hasil.append(n.status); n = n.induk
        return hasil[::-1]

def ekspansi(masalah, node):
    """Membangkitkan semua node anak dari sebuah node."""
    for a in masalah.aksi(node.status):
        s2 = masalah.hasil(node.status, a)
        yield Node(s2, node, a, node.g + masalah.biaya(node.status, a, s2))

class Hasil:
    """Ringkasan satu kali pencarian."""
    def __init__(self, nama, node, urutan, frontier_maks):
        self.nama = nama
        self.jalur = node.jalur() if node else None
        self.biaya = node.g if node else None
        self.urutan_ekspansi = urutan
        self.frontier_maks = frontier_maks
    def __repr__(self):
        if self.jalur is None:
            return f"{self.nama}: tidak ditemukan ({len(self.urutan_ekspansi)} node diekspansi)"
        return (f"{self.nama}: {' → '.join(map(str, self.jalur))} | biaya {self.biaya} | "
                f"{len(self.urutan_ekspansi)} node diekspansi | frontier maks {self.frontier_maks}")

## 3. Breadth-First Search (BFS)
BFS mengekspansi node **paling dangkal** lebih dulu: seluruh node kedalaman 1, lalu kedalaman 2, dan seterusnya.

> **Konvensi mata kuliah ini:** uji tujuan dilakukan saat node **diambil dari frontier** (bukan saat dibangkitkan), dan aksi dicoba menurut urutan alfabet. Konvensi yang sama dipakai pada telusur manual di tugas dan ujian.

In [ ]:
def bfs(masalah, jejak=False):
    awal = Node(masalah.awal)
    frontier = deque([awal]); reached = {awal.status}
    urutan, fmaks = [], 1
    while frontier:
        if jejak:
            print(f"frontier: {[n.status for n in frontier]}")
        node = frontier.popleft()                       # FIFO: ambil yang paling lama
        urutan.append(node.status)
        if jejak: print(f"   ekspansi → {node.status}")
        if masalah.tujuan(node.status):
            return Hasil("BFS", node, urutan, fmaks)
        for anak in ekspansi(masalah, node):
            if anak.status not in reached:
                reached.add(anak.status); frontier.append(anak)
        fmaks = max(fmaks, len(frontier))
    return Hasil("BFS", None, urutan, fmaks)

h_bfs = bfs(rute, jejak=True)
print(); print(h_bfs)

BFS menemukan rute dengan **jumlah ruas paling sedikit** (3 ruas), tetapi belum tentu yang jaraknya terpendek. BFS tidak melihat bobot sisi sama sekali.

## 4. Depth-First Search (DFS)
DFS mengekspansi node **paling dalam** lebih dulu. Frontier berupa tumpukan (stack).

In [ ]:
def dfs(masalah, jejak=False):
    frontier = [Node(masalah.awal)]; dikunjungi = set()
    urutan, fmaks = [], 1
    while frontier:
        node = frontier.pop()                           # LIFO: ambil yang paling baru
        if node.status in dikunjungi:
            continue
        dikunjungi.add(node.status); urutan.append(node.status)
        if jejak: print(f"ekspansi → {node.status:16s} (kedalaman {node.kedalaman})")
        if masalah.tujuan(node.status):
            return Hasil("DFS", node, urutan, fmaks)
        anak2 = [a for a in ekspansi(masalah, node) if a.status not in dikunjungi]
        frontier.extend(reversed(anak2))                # dibalik supaya anak pertama (alfabet) diambil lebih dulu
        fmaks = max(fmaks, len(frontier))
    return Hasil("DFS", None, urutan, fmaks)

h_dfs = dfs(rute, jejak=True)
print(); print(h_dfs)

Kali ini DFS **beruntung**: dengan urutan alfabet, selaman pertamanya langsung mengarah ke Payakumbuh. Keberuntungan itu tidak bisa diandalkan. Coba pasangan kota lain:

In [ ]:
rute_lain = MasalahRute(G, "Pariaman", "Solok")
print(dfs(rute_lain)); print(bfs(rute_lain))

Untuk Pariaman → Solok, DFS menyelam ke arah yang salah dan baru sampai setelah berkeliling ratusan kilometer. DFS **tidak optimal**: ia mengembalikan solusi pertama yang ditemui, apa pun mutunya. Keunggulannya ada pada memori: pada ruang keadaan berbentuk pohon, DFS hanya perlu menyimpan satu jalur beserta saudara-saudaranya, yaitu O(bm).

## 5. Depth-Limited Search (DLS) dan Iterative Deepening Search (IDS)
- **DLS**: DFS yang berhenti menyelam pada batas kedalaman `l`. Aman dari jalur tak hingga, tetapi gagal bila tujuan lebih dalam dari `l`.
- **IDS**: menjalankan DLS berulang dengan `l = 0, 1, 2, ...` sampai tujuan ditemukan. Memadukan hemat memori ala DFS dengan jaminan solusi terdangkal ala BFS.

In [ ]:
def dls(masalah, batas):
    """DFS berbatas kedalaman (versi pohon dengan pemeriksaan siklus pada jalur)."""
    urutan = []
    def rekursi(node):
        urutan.append(node.status)
        if masalah.tujuan(node.status): return node
        if node.kedalaman == batas: return None
        di_jalur = set(node.jalur())
        for anak in ekspansi(masalah, node):
            if anak.status in di_jalur: continue        # jangan kembali ke status yang sudah ada di jalur ini
            temuan = rekursi(anak)
            if temuan: return temuan
        return None
    return Hasil(f"DLS(l={batas})", rekursi(Node(masalah.awal)), urutan, batas + 1)

def ids(masalah, batas_maks=50):
    total = []
    for batas in range(batas_maks + 1):
        h = dls(masalah, batas)
        total += h.urutan_ekspansi
        print(f"  batas {batas}: {len(h.urutan_ekspansi):3d} node dikunjungi →", "DITEMUKAN" if h.jalur else "belum")
        if h.jalur:
            h.nama = "IDS"; h.urutan_ekspansi = total
            return h
    return Hasil("IDS", None, total, batas_maks)

print(dls(rute, 2)); print()
h_ids = ids(rute)
print(); print(h_ids)

IDS mengunjungi ulang node-node dangkal pada setiap iterasi. Kelihatannya boros, tetapi karena jumlah node tumbuh eksponensial terhadap kedalaman, sebagian besar pekerjaan selalu berada di lapis terakhir. Pemborosannya kecil.

## 6. Uniform-Cost Search (UCS)
UCS mengekspansi node dengan **biaya jalur `g` terkecil**. Frontier berupa antrean prioritas. Bila semua biaya langkah positif, UCS dijamin menemukan solusi **berbiaya minimum**.

In [ ]:
def ucs(masalah, jejak=False):
    awal = Node(masalah.awal)
    hitung = 0                                           # pemecah seri agar heap tidak membandingkan Node
    frontier = [(0, hitung, awal)]
    g_terbaik = {}                                       # status -> g terkecil saat diekspansi
    urutan, fmaks = [], 1
    while frontier:
        g, _, node = heapq.heappop(frontier)             # ambil g terkecil
        if node.status in g_terbaik:
            continue                                     # sudah pernah diekspansi lewat jalur yang lebih murah
        g_terbaik[node.status] = g; urutan.append(node.status)
        if jejak: print(f"ekspansi → {node.status:16s} g = {g}")
        if masalah.tujuan(node.status):
            return Hasil("UCS", node, urutan, fmaks)
        for anak in ekspansi(masalah, node):
            if anak.status not in g_terbaik:
                hitung += 1
                heapq.heappush(frontier, (anak.g, hitung, anak))
        fmaks = max(fmaks, len(frontier))
    return Hasil("UCS", None, urutan, fmaks)

h_ucs = ucs(rute, jejak=True)
print(); print(h_ucs)

Perhatikan nilai `g` yang **tidak pernah turun** sepanjang urutan ekspansi. Itulah inti UCS: ia menyapu ruang keadaan dalam "lingkaran biaya" yang makin melebar, sehingga saat tujuan diambil dari frontier, tidak mungkin ada jalur lain yang lebih murah.

## 7. Membandingkan hasil

In [ ]:
hasil_semua = [h_bfs, h_dfs, h_ids, h_ucs]
tabel = pd.DataFrame([{
    "algoritma": h.nama,
    "jalur": " → ".join(h.jalur),
    "jumlah ruas": len(h.jalur) - 1,
    "jarak (km)": h.biaya,
    "node diekspansi": len(h.urutan_ekspansi),
    "frontier maks": h.frontier_maks,
} for h in hasil_semua])
pd.set_option("display.max_colwidth", 120); pd.set_option("display.width", 250)
print(tabel.to_string(index=False))

In [ ]:
def gambar_jalur(daftar_hasil):
    fig, sumbu = plt.subplots(1, len(daftar_hasil), figsize=(5.2 * len(daftar_hasil), 5.6))
    for ax, h in zip(sumbu, daftar_hasil):
        nx.draw_networkx_nodes(G, POSISI, ax=ax, node_size=900, node_color="#E6F2F1", edgecolors="#12263A")
        nx.draw_networkx_labels(G, POSISI, ax=ax, font_size=6.5, labels={k: k.replace(" ", "\n") for k in G})
        nx.draw_networkx_edges(G, POSISI, ax=ax, edge_color="#8A9AA5")
        nx.draw_networkx_edges(G, POSISI, ax=ax, edgelist=list(zip(h.jalur, h.jalur[1:])), edge_color="#D1495B", width=4)
        ax.set_title(f"{h.nama}: {h.biaya} km"); ax.axis("off")
    plt.tight_layout(); plt.show()

gambar_jalur(hasil_semua)

### Ringkasan sifat (b = faktor percabangan, d = kedalaman solusi terdangkal, m = kedalaman maksimum, C* = biaya solusi optimal, ε = biaya langkah terkecil)

| Kriteria | BFS | DFS | DLS | IDS | UCS |
|---|---|---|---|---|---|
| Lengkap (*complete*)? | Ya | Tidak (pohon tak hingga) | Tidak (bila l < d) | Ya | Ya |
| Optimal? | Ya, bila biaya langkah seragam | Tidak | Tidak | Ya, bila biaya langkah seragam | Ya |
| Waktu | O(b^d) | O(b^m) | O(b^l) | O(b^d) | O(b^(1+⌊C*/ε⌋)) |
| Memori | O(b^d) | O(bm) | O(bl) | O(bd) | O(b^(1+⌊C*/ε⌋)) |

## 8. Eksperimen: seberapa cepat pencarian membengkak?
Kita ukur pada 8-puzzle. Susunan acak dibuat dengan mengacak dari status tujuan, lalu kita catat jumlah node yang diekspansi BFS dan IDS untuk berbagai kedalaman solusi.

In [ ]:
class Puzzle8(Masalah):
    TUJUAN = (1, 2, 3, 4, 5, 6, 7, 8, 0)
    GESER = {"Atas": -3, "Bawah": 3, "Kiri": -1, "Kanan": 1}
    def __init__(self, awal): super().__init__(tuple(awal))
    def aksi(self, s):
        baris, kolom = divmod(s.index(0), 3); p = []
        if baris > 0: p.append("Atas")
        if baris < 2: p.append("Bawah")
        if kolom > 0: p.append("Kiri")
        if kolom < 2: p.append("Kanan")
        return p
    def hasil(self, s, a):
        i = s.index(0); j = i + self.GESER[a]; papan = list(s)
        papan[i], papan[j] = papan[j], papan[i]; return tuple(papan)
    def tujuan(self, s): return s == self.TUJUAN

def acak_puzzle(langkah, seed):
    rng = random.Random(seed); p = Puzzle8(Puzzle8.TUJUAN); s = p.awal; sebelum = None
    for _ in range(langkah):
        pilihan = [p.hasil(s, a) for a in p.aksi(s)]
        pilihan = [x for x in pilihan if x != sebelum] or pilihan
        sebelum, s = s, rng.choice(pilihan)
    return s

def ids_senyap(masalah, batas_maks=30):
    total = 0
    for batas in range(batas_maks + 1):
        h = dls(masalah, batas); total += len(h.urutan_ekspansi)
        if h.jalur: return len(h.jalur) - 1, total
    return None, total

baris = []
for langkah, seed in [(2, 1), (4, 2), (6, 3), (8, 4), (10, 5), (12, 6), (14, 7)]:
    p = Puzzle8(acak_puzzle(langkah, seed))
    hb = bfs(p); d = len(hb.jalur) - 1
    _, n_ids = ids_senyap(p)
    baris.append({"kedalaman solusi d": d, "BFS: node diekspansi": len(hb.urutan_ekspansi),
                  "BFS: frontier maks": hb.frontier_maks, "IDS: node dikunjungi": n_ids})
eksperimen = pd.DataFrame(baris).drop_duplicates("kedalaman solusi d").sort_values("kedalaman solusi d")
print(eksperimen.to_string(index=False))

In [ ]:
plt.figure(figsize=(8, 4.2))
plt.plot(eksperimen["kedalaman solusi d"], eksperimen["BFS: node diekspansi"], "o-", color="#0E7C86", label="BFS: node diekspansi")
plt.plot(eksperimen["kedalaman solusi d"], eksperimen["IDS: node dikunjungi"], "s-", color="#D1495B", label="IDS: node dikunjungi")
plt.plot(eksperimen["kedalaman solusi d"], eksperimen["BFS: frontier maks"], "^--", color="#F2A024", label="BFS: frontier maks (memori)")
plt.yscale("log"); plt.xlabel("kedalaman solusi d"); plt.ylabel("jumlah node (skala log)")
plt.title("8-puzzle: biaya pencarian tumbuh eksponensial terhadap kedalaman"); plt.legend(); plt.grid(alpha=0.3); plt.show()

Garis yang hampir lurus pada **skala logaritmik** berarti pertumbuhannya eksponensial. Setiap tambahan beberapa langkah kedalaman melipatgandakan pekerjaan. IDS versi pohon mengunjungi lebih banyak node daripada BFS (karena tidak mengingat status yang sudah dicapai), tetapi memorinya hanya sepanjang jalur. Tanpa informasi tambahan, tidak ada strategi yang lolos dari ledakan ini. Minggu depan kita memberi pencarian **petunjuk arah**: heuristik.

---
## Latihan (tahap 1 asesmen A2)

### Latihan 1 — Telusur manual lalu cocokkan dengan kode
Graf T pada lembar tugas pertemuan 4 memiliki sisi berbobot berikut:

`S–A 3, S–B 6, A–C 5, A–D 4, B–D 4, B–E 4, C–D 2, C–G 7, D–G 4, E–G 9`

1. Kerjakan dulu telusur BFS, DFS, dan UCS dari S ke G **di kertas** (lembar tugas).
2. Bangun graf T pada sel di bawah, jalankan ketiga algoritma, lalu bandingkan dengan hasil kertas Anda. Bila berbeda, jelaskan letak kekeliruannya.

In [ ]:
T = nx.Graph()
# TODO: tambahkan sisi graf T, contoh: T.add_edge("S", "A", jarak=3)

if T.number_of_edges() > 0:
    masalah_t = MasalahRute(T, "S", "G")
    for algoritma in (bfs, dfs, ucs):
        h = algoritma(masalah_t)
        print(h); print("   urutan ekspansi:", h.urutan_ekspansi)
else:
    print("Graf T belum diisi.")

### Latihan 2 — Urutan aksi memengaruhi hasil
Ubah `MasalahRute.aksi` agar mengembalikan tetangga dalam urutan alfabet **terbalik** (buat kelas turunan, jangan mengubah kelas asli). Jalankan BFS, DFS, dan UCS dari Padang ke Payakumbuh.
- Algoritma mana yang hasilnya berubah? Mana yang tidak? Mengapa?

In [ ]:
# Jawaban Latihan 2

### Latihan 3 — BFS dengan uji tujuan lebih awal
Tulis `bfs_awal(masalah)`: sama seperti BFS, tetapi uji tujuan dilakukan **saat node dibangkitkan**. Bandingkan jumlah node yang diekspansi dengan `bfs` pada rute Padang → Payakumbuh dan pada tiga susunan 8-puzzle.
- Apakah jalur yang ditemukan berubah?
- Mengapa trik yang sama **tidak boleh** dipakai pada UCS?

In [ ]:
# Jawaban Latihan 3

### Refleksi
1. Untuk aplikasi navigasi, algoritma mana dari kelima strategi ini yang paling layak? Jelaskan dengan bukti dari tabel perbandingan.
2. Kapan DFS justru menjadi pilihan yang masuk akal?
3. Berdasarkan grafik eksperimen, taksir jumlah node yang diekspansi BFS bila kedalaman solusi 20. Apa artinya bagi memori?

*Jawaban:*

---
**Pengumpulan:** simpan notebook ini. Notebook pertemuan 4 dan 5 dikumpulkan **bersama** sebagai artefak A2 setelah pertemuan 5.